# EXP-021 원인 분석: 거부 감소, 모호·회피 증가, judge 형식 오류

사용자 지시(2026-11-03)로 판정 뒤에 쓴다. 새 API 호출이 없고 저장된 기록만 읽는다. 질의 원문과 근거 원문은 출력하지 않는다.

In [1]:
import json
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
ROOT = Path.cwd().parent
PREFIX = "hybrid-cond-filter-law-ms256-k20-ce"
RUNS = {"base": "v4-dev-full-spk-head", "cand": "v8-dev-full-spk-marks"}
JUDGE = "deepseek_go_mixed-j_v13"
AMB = ("모호", "회피")

In [3]:
def read(path):
    with open(path, encoding="utf-8") as f:
        return {r["qid"]: r for r in map(json.loads, f)}


def state(r):
    """report의 근거 상태와 같은 구분."""
    if r.get("gold_n") is None:
        return "정답 문서 없음"
    inc = r.get("inclusion") or {}
    if inc.get("excluded") == "reference_unusable":
        return "참조 사용 불가"
    return "청크에 답 있음" if r.get("answer_in_chunk") else "청크에 답 없음"


def items(r):
    return (r.get("item_eval") or {}).get("items") or []

## 데이터 로딩

In [4]:
judge = {k: read(ROOT / f"data/gen_eval/judge/{PREFIX}-{v}-{JUDGE}.jsonl") for k, v in RUNS.items()}
gens = {k: read(ROOT / f"data/gen_eval/runs/{PREFIX}-{v}.jsonl") for k, v in RUNS.items()}
qids = list(judge["base"])
print(len(qids), set(qids) == set(judge["cand"]))

330 True


## 1. 거부 감소

기준에서 전체 거부였던 질의가 후보에서 어떻게 바뀌었는지 본다.

In [5]:
base_ref = pd.Series([str(judge["base"][q]["refusal"]) for q in qids], name="기준")
cand_ref = pd.Series([str(judge["cand"][q]["refusal"]) for q in qids], name="후보")
pd.crosstab(base_ref, cand_ref)

후보,None,full,partial
기준,,,
None,274,0,6
full,24,3,1
partial,11,0,11


In [6]:
moved = [q for q in qids if judge["base"][q]["refusal"] == "full" and judge["cand"][q]["refusal"] != "full"]
rows = []
for q in moved:
    b, c = judge["base"][q], judge["cand"][q]
    f, rel = c.get("faithfulness") or {}, c.get("relevance") or {}
    rows.append({"qid": q, "기준 상태": state(b), "후보 상태": state(c), "후보 거부": c["refusal"],
                 "충실성": f.get("score"), "관련성": rel.get("label"),
                 "처리": dict(Counter(i.get("outcome") for i in items(c)))})
moved_df = pd.DataFrame(rows)
print(len(moved_df))
moved_df

25


,qid,기준 상태,후보 상태,후보 거부,충실성,관련성,처리
0,051401-0006,청크에 답 없음,청크에 답 있음,NaN,1.00,답함,{'근거 없이 답함': 1}
1,049487-0012,청크에 답 없음,참조 사용 불가,NaN,1.00,답하지 않음,{'무언 생략': 2}
2,046608-0011,청크에 답 있음,청크에 답 있음,NaN,1.00,답함,{'맞음': 1}
3,051550-0018,청크에 답 없음,참조 사용 불가,NaN,1.00,답함,{'상황 미전달': 1}
4,047672-0008,청크에 답 있음,청크에 답 있음,NaN,1.00,답함,{'상황 미전달': 1}
5,049763-0012,청크에 답 없음,청크에 답 있음,NaN,1.00,일부,{'근거 없이 답함': 1}
6,048590-0004,청크에 답 없음,청크에 답 없음,NaN,0.75,답함,"{'안내 없이 부분 답함': 1, '무언 생략': 1}"
7,032409-0001,청크에 답 없음,청크에 답 있음,NaN,1.00,답함,{'상황 전달': 1}
8,048873-0011,청크에 답 있음,청크에 답 있음,NaN,1.00,답함,{'과잉 거부': 1}
9,050287-0001,청크에 답 없음,청크에 답 없음,NaN,1.00,답하지 않음,{'맞음': 1}


In [7]:
print("기준 상태", moved_df["기준 상태"].value_counts().to_dict())
print("후보 상태", moved_df["후보 상태"].value_counts().to_dict())
print("상태가 바뀐 질의", (moved_df["기준 상태"] != moved_df["후보 상태"]).sum())
print("후보 거부", moved_df["후보 거부"].fillna("없음").value_counts().to_dict())
print("충실성 평균", round(moved_df["충실성"].mean(), 3), "1.0 미만", (moved_df["충실성"] < 1).sum())
print("관련성", moved_df["관련성"].value_counts(dropna=False).to_dict())
all_out = Counter()
for q in moved:
    all_out.update(i.get("outcome") for i in items(judge["cand"][q]))
print("항목 처리", all_out.most_common())

기준 상태 {'청크에 답 없음': 18, '청크에 답 있음': 6, '정답 문서 없음': 1}
후보 상태 {'청크에 답 있음': 14, '참조 사용 불가': 5, '청크에 답 없음': 5, '정답 문서 없음': 1}
상태가 바뀐 질의 13
후보 거부 {'없음': 24, 'partial': 1}
충실성 평균 0.948 1.0 미만 4
관련성 {'답함': 16, '일부': 6, '답하지 않음': 3}
항목 처리 [('맞음', 12), ('근거 없이 답함', 6), ('무언 생략', 5), ('상황 미전달', 5), ('안내 없이 부분 답함', 3), ('상황 전달', 2), ('과잉 거부', 1), ('불필요한 부분 안내', 1)]


In [8]:
gold_absent = [q for q in qids if judge["cand"][q].get("gold_n") is None]
print("정답 문서 없음", len(gold_absent))
for q in gold_absent:
    c = judge["cand"][q]
    print(q, "거부", judge["base"][q]["refusal"], "->", c["refusal"],
          "| 정보 상태", Counter(i.get("state") for i in items(c)).most_common(),
          "| 처리", Counter(i.get("outcome") for i in items(c)).most_common())

정답 문서 없음 11
046280-0008 거부 None -> None | 정보 상태 [('있음', 1)] | 처리 [('맞음', 1)]
049848-0012 거부 None -> None | 정보 상태 [('있음', 1), ('일부', 1)] | 처리 [('맞음', 1), ('안내 없이 부분 답함', 1)]
043344-0001 거부 partial -> partial | 정보 상태 [('없음', 1)] | 처리 [('맞음', 1)]
050738-0012 거부 None -> None | 정보 상태 [('있음', 1)] | 처리 [('맞음', 1)]
047535-0009 거부 None -> partial | 정보 상태 [('있음', 1), ('없음', 1)] | 처리 [('과잉 거부', 1), ('근거 없이 답함', 1)]
052412-0003 거부 None -> None | 정보 상태 [('있음', 1)] | 처리 [('맞음', 1)]
051405-0013 거부 None -> None | 정보 상태 [('일부', 1), ('있음', 1)] | 처리 [('안내 없이 부분 답함', 1), ('맞음', 1)]
051405-0025 거부 None -> None | 정보 상태 [('없음', 1)] | 처리 [('근거 없이 답함', 1)]
046687-0013 거부 None -> None | 정보 상태 [('있음', 1)] | 처리 [('맞음', 1)]
048620-0009 거부 full -> None | 정보 상태 [('없음', 2)] | 처리 [('맞음', 2)]
051938-0010 거부 None -> None | 정보 상태 [('있음', 1)] | 처리 [('과잉 거부', 1)]


## 2. 모호·회피 증가

항목은 judge가 근거를 보고 새로 나누므로 두 실행의 항목이 1:1로 대응하지 않는다. 질의 단위로 건수 이동을 본다.

In [9]:
for run in judge:
    all_items = [i for q in qids for i in items(judge[run][q])]
    amb = [i for i in all_items if i.get("state") in AMB]
    print(run, "정보 상태", Counter(i.get("state") for i in all_items).most_common())
    print(run, "모호·회피", len(amb), "처리", Counter(i.get("handling") for i in amb).most_common())
    print(run, "모호·회피 결과", Counter(i.get("outcome") for i in amb).most_common())

base 정보 상태 [('있음', 212), ('없음', 142), ('일부', 76), ('모호', 39), ('회피', 12)]
base 모호·회피 51 처리 [('상황을 밝히고 답함', 20), ('답함', 19), ('다루지 않음', 6), ('거부', 5), ('부분임을 밝히고 답함', 1)]
base 모호·회피 결과 [('상황 미전달', 31), ('상황 전달', 20)]
cand 정보 상태 [('있음', 232), ('없음', 108), ('일부', 64), ('모호', 53), ('회피', 22)]
cand 모호·회피 75 처리 [('상황을 밝히고 답함', 44), ('답함', 20), ('다루지 않음', 9), ('거부', 2)]
cand 모호·회피 결과 [('상황 전달', 44), ('상황 미전달', 31)]


In [10]:
rows = []
for q in qids:
    b, c = items(judge["base"][q]), items(judge["cand"][q])
    rows.append({"qid": q,
                 "기준": sum(i.get("state") in AMB for i in b),
                 "후보": sum(i.get("state") in AMB for i in c),
                 "기준 거부": judge["base"][q]["refusal"],
                 "1위가 정답": judge["cand"][q].get("gold_n") == 1,
                 "기준 답 있음": bool(judge["base"][q].get("answer_in_chunk")),
                 "후보 답 있음": bool(judge["cand"][q].get("answer_in_chunk")),
                 "기준 항목": len(b), "후보 항목": len(c)})
amb_df = pd.DataFrame(rows)
up = amb_df[amb_df["후보"] > amb_df["기준"]]
down = amb_df[amb_df["후보"] < amb_df["기준"]]
print("모호·회피 늘어난 질의", len(up), "줄어든 질의", len(down), "같은 질의", len(amb_df) - len(up) - len(down))
print("늘어난 질의 중 기준 전체 거부", (up["기준 거부"] == "full").sum(), "| 1위가 정답", up["1위가 정답"].sum())
print("늘어난 질의 중 답 없음에서 있음으로", ((~up["기준 답 있음"]) & up["후보 답 있음"]).sum())
print("줄어든 질의 중 1위가 정답", down["1위가 정답"].sum())
print("양쪽 모두 있는 질의", ((amb_df["기준"] > 0) & (amb_df["후보"] > 0)).sum())
up.head(30)

모호·회피 늘어난 질의 38 줄어든 질의 18 같은 질의 274
늘어난 질의 중 기준 전체 거부 6 | 1위가 정답 38
늘어난 질의 중 답 없음에서 있음으로 8
줄어든 질의 중 1위가 정답 16
양쪽 모두 있는 질의 30


,qid,기준,후보,기준 거부,1위가 정답,기준 답 있음,후보 답 있음,기준 항목,후보 항목
9,048610-0013,1,2,NaN,True,False,False,2,2
14,044259-0001,0,1,NaN,True,False,True,2,2
21,047927-0006,0,1,NaN,True,True,True,1,1
25,048610-0012,0,1,NaN,True,True,True,2,2
28,051550-0018,0,1,full,True,False,False,1,1
33,049562-0015,0,2,NaN,True,False,False,2,2
57,028778-0001,0,1,NaN,True,True,True,1,1
60,047672-0008,0,1,full,True,True,True,1,1
73,048873-0013,0,2,NaN,True,False,False,2,2
74,049211-0005,0,1,NaN,True,True,True,2,2


In [11]:
both = amb_df[(amb_df["기준"] > 0) & (amb_df["후보"] > 0)]["qid"]
for run in judge:
    amb = [i for q in both for i in items(judge[run][q]) if i.get("state") in AMB]
    print(run, "양쪽 모두 있는 질의의 항목", len(amb), Counter(i.get("outcome") for i in amb).most_common())
new_amb = [i for q in up["qid"] for i in items(judge["cand"][q]) if i.get("state") in AMB]
print("늘어난 질의의 후보 항목", len(new_amb), Counter(i.get("outcome") for i in new_amb).most_common())

base 양쪽 모두 있는 질의의 항목 33 [('상황 전달', 17), ('상황 미전달', 16)]
cand 양쪽 모두 있는 질의의 항목 38 [('상황 전달', 24), ('상황 미전달', 14)]
늘어난 질의의 후보 항목 47 [('상황 전달', 27), ('상황 미전달', 20)]


## 3. judge 형식 오류와 끝나지 않은 응답

In [12]:
def statement_count(r):
    return len((r.get("faithfulness") or {}).get("verdicts") or [])


def usage_list(f):
    u = f.get("usage")
    return u if isinstance(u, list) else [u or {}]


rows = []
for run in judge:
    for q in qids:
        f = judge[run][q].get("faithfulness")
        if f is None or f.get("score") is not None or f.get("excluded") or f.get("error") == "진술 없음":
            continue
        g = gens[run][q]
        rows.append({"실행": run, "qid": q, "오류": str(f.get("error"))[:80], "답변 길이": len(g["answer"]),
                     "근거 문자": sum(len(c["text"]) for c in g["contexts"]),
                     "응답 토큰": [u.get("completion_tokens") for u in usage_list(f)],
                     "끝난 이유": [u.get("finish_reason") for u in usage_list(f)]})
err_df = pd.DataFrame(rows)
err_df

,실행,qid,오류,답변 길이,근거 문자,응답 토큰,끝난 이유
0,base,028778-0001,verdicts 형식 오류,198,2233,"[134, 1614, 1339]","[stop, stop, stop]"
1,base,046533-0016,verdicts 형식 오류,356,3175,"[263, 1877, 1245]","[stop, stop, stop]"
2,base,048701-0005,verdicts 형식 오류,98,2653,"[81, 2552, 1811]","[stop, stop, stop]"
3,base,047673-0001,verdicts 형식 오류,202,3391,"[156, 1361, 625]","[stop, stop, stop]"
4,cand,048610-0013,verdicts 형식 오류,348,3890,"[233, 1952, 1017]","[stop, stop, stop]"
5,cand,048338-0012,verdicts 형식 오류,365,3540,"[268, 3705, 918]","[stop, stop, stop]"
6,cand,049763-0013,verdicts 형식 오류,293,4404,"[179, 4298, 1239]","[stop, stop, stop]"
7,cand,050619-0010,verdicts 형식 오류,404,6003,"[280, 1552, 961]","[stop, stop, stop]"
8,cand,049460-0012,verdicts 형식 오류,209,3807,"[145, 1774, 755]","[stop, stop, stop]"


In [13]:
for run in judge:
    lengths = np.array([len(gens[run][q]["answer"]) for q in qids])
    counts = [statement_count(judge[run][q]) for q in qids if statement_count(judge[run][q])]
    print(run, "답변 길이 중앙값", int(np.median(lengths)), "p95", int(np.percentile(lengths, 95)), "| 진술 수 평균", round(np.mean(counts), 2), "최대", max(counts))
base_err, cand_err = [set(err_df[err_df["실행"] == r]["qid"]) for r in ("base", "cand")]
print("양쪽에서 함께 오류인 질의", base_err & cand_err)

base 답변 길이 중앙값 155 p95 414 | 진술 수 평균 2.51 최대 8
cand 답변 길이 중앙값 200 p95 561 | 진술 수 평균 2.76 최대 10
양쪽에서 함께 오류인 질의 set()


## 4. 지연

기록만 남긴다. 일회성인지 반복되는지는 다음 실행에서 본다(사용자 지시 2026-11-03).

In [14]:
for run in gens:
    first = np.array([gens[run][q]["first_token_s"] for q in qids])
    prompt = np.array([gens[run][q]["usage"]["prompt_tokens"] for q in qids])
    print(run, "첫 토큰 중앙값", round(np.median(first), 2), "p95", round(np.percentile(first, 95), 2), "3초 초과", (first > 3).sum(), "| 입력 토큰 중앙값", int(np.median(prompt)), "| 상관", round(np.corrcoef(prompt, first)[0, 1], 2))

base 첫 토큰 중앙값 0.68 p95 1.57 3초 초과 3 | 입력 토큰 중앙값 2332 | 상관 0.06
cand 첫 토큰 중앙값 0.89 p95 4.38 3초 초과 29 | 입력 토큰 중앙값 3031 | 상관 0.14
